# 02 - Clean Wuzzuf Jobs

Stage 2 of the pipeline.

1. Reads **only** `PENDING_RAW_QUEUE` — the batch of jobs notebook 1 scraped since this notebook last ran. If it's empty, there's nothing new to clean and the notebook stops after saying so.
2. Standardizes every field in that batch: pulls the English translation out of Wuzzuf's localized objects, turns the HTML description into plain bullet lines, formats experience/salary/dates into consistent strings.
3. Appends the cleaned batch onto `CLEAN_HISTORY` (deduplicated by `job_url`, so re-running this notebook on the same batch is safe) and onto `PENDING_CLEANED_QUEUE` for notebook 3.
4. Clears `PENDING_RAW_QUEUE` now that it's been consumed.

In [ ]:
import json
import os
import logging
from datetime import datetime, timezone

## Settings

In [ ]:
PENDING_RAW_QUEUE = "../data/raw/wuzzuf_pending_raw_queue.json"       # input: unconsumed batch from notebook 1
CLEAN_HISTORY = "../data/interim/wuzzuf_jobs_cleaned.json"           # output: full cleaned history (audit)
PENDING_CLEANED_QUEUE = "../data/interim/wuzzuf_pending_cleaned_queue.json"  # output: unconsumed batch for notebook 3

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("wuzzuf.clean")

## Queue / history helpers

In [ ]:
def load_json(path):
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(records, path):
    dirname = os.path.dirname(path)
    if dirname:
        os.makedirs(dirname, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def dedupe_keep_last(records, key="job_url"):
    """Merge records, keeping the most recent version of each key."""
    deduped = {}
    for r in records:
        k = r.get(key)
        if k is None:
            continue
        deduped[k] = r
    return list(deduped.values())

## Load the pending batch

In [ ]:
pending_raw = load_json(PENDING_RAW_QUEUE)
log.info("Pending raw queue: %d job(s) to clean", len(pending_raw))

## Field-level cleaning helpers

In [ ]:
def localized(node, field, fallback_field=None):
    """Extract the English translation from a Wuzzuf localized object."""
    if not isinstance(node, dict):
        return None

    translations = node.get("translations") or {}
    field_translations = translations.get(field)

    if isinstance(field_translations, dict) and field_translations.get("en"):
        return field_translations["en"]

    if node.get(field):
        return node[field]

    if fallback_field and node.get(fallback_field):
        return node[fallback_field]

    return None

In [ ]:
def html_to_bullets(html_text):
    """Convert an HTML job description into a list of plain-text lines."""
    if not html_text:
        return []

    from bs4 import BeautifulSoup

    soup = BeautifulSoup(html_text, "html.parser")

    items = soup.find_all("li")
    if items:
        bullets = [li.get_text(" ", strip=True) for li in items if li.get_text(strip=True)]
        if bullets:
            return bullets

    text = soup.get_text("\n", strip=True)
    return [line for line in text.split("\n") if line]

In [ ]:
def format_experience(years_obj):
    """Convert a Wuzzuf experience object into '1-2 years' / '3+ years' etc."""
    if not isinstance(years_obj, dict):
        return None

    lo = years_obj.get("min")
    hi = years_obj.get("max")

    if lo is None and hi is None:
        return None
    if lo is None:
        return f"0 - {hi} years"
    if hi is None:
        return f"{lo}+ years"
    if lo == hi:
        return f"{lo} years"
    return f"{lo}-{hi} years"

In [ ]:
def format_salary(salary_obj):
    """Convert a Wuzzuf salary object into a single standardized string."""
    if not isinstance(salary_obj, dict):
        return None

    if not salary_obj.get("isPaid", True):
        return "Unpaid"

    lo = salary_obj.get("min")
    hi = salary_obj.get("max")

    currency = salary_obj.get("currency") or {}
    currency_code = currency.get("code") or localized(currency, "name")

    period = salary_obj.get("period") or {}
    period_text = localized(period, "period") or period.get("name")

    extra = salary_obj.get("additionalDetails")

    if lo is None and hi is None:
        base = "Confidential"
    elif lo is not None and hi is not None:
        base = f"{lo:,}-{hi:,}"
    else:
        base = f"{lo or hi:,}"

    parts = [base]
    if currency_code and base != "Confidential":
        parts.append(currency_code)
    if period_text and base != "Confidential":
        parts.append(period_text)

    text = " ".join(parts)
    if extra:
        text = f"{text} ({extra})"

    return text

In [ ]:
def format_timestamp(value):
    """Convert a Wuzzuf timestamp into 'YYYY-MM-DD HH:MM:SS' (UTC)."""
    if not value:
        return None

    try:
        value = str(value).replace("Z", "+00:00")
        dt = datetime.fromisoformat(value)

        if dt.tzinfo is not None:
            dt = dt.astimezone(timezone.utc).replace(tzinfo=None)

        return dt.strftime("%Y-%m-%d %H:%M:%S")

    except (ValueError, TypeError):
        return str(value)

## Clean one record

In [ ]:
def clean_record(raw_record):
    """Turn one raw scraped record into a cleaned, standardized dict."""
    attrs = raw_record.get("job_attributes") or {}
    company_attrs = raw_record.get("company_attributes") or {}

    if not attrs:
        return None

    user_translations = attrs.get("userContentTranslations", {})
    title = (user_translations.get("title") or {}).get("en") or attrs.get("title")

    company_name = company_attrs.get("name")

    location = attrs.get("location") or {}
    city = localized(location.get("city") or {}, "name")
    country = localized(location.get("country") or {}, "name")

    workplace = attrs.get("workplaceArrangement") or {}
    work_mode = localized(workplace, "displayedName")

    work_types = attrs.get("workTypes") or []
    employment_type = ", ".join(
        filter(None, (localized(wt, "displayedName") for wt in work_types))
    ) or None

    career_level = attrs.get("careerLevel") or {}
    career_level_text = localized(career_level, "displayName") or localized(career_level, "name")

    education_level = attrs.get("candidatePreferences", {}).get("educationLevel") or {}
    qualification = localized(education_level, "name")

    experience_needed = format_experience(attrs.get("workExperienceYears"))

    keywords = attrs.get("keywords") or []
    skills = [k.get("name") for k in keywords if k.get("name")]

    salary = format_salary(attrs.get("salary"))

    description_list = html_to_bullets(attrs.get("description"))
    job_description = "\n".join(description_list) if description_list else None

    posted_date = format_timestamp(attrs.get("postedAt"))

    return {
        "job_id": raw_record.get("job_id"),
        "job_title": title,
        "company_name": company_name,
        "job_description": job_description,
        "job_url": raw_record.get("job_url"),
        "country": country,
        "city": city,
        "work_mode": work_mode,
        "employment_type": employment_type,
        "career_level": career_level_text,
        "experience_needed": experience_needed,
        "qualification_required": qualification,
        "skills": skills,
        "salary": salary,
        "posted_date": posted_date,
    }

## Clean the pending batch, update history + downstream queue

Everything here is a no-op (with a log message) if `pending_raw` is empty, so it's always safe to run this notebook even when notebook 1 hasn't produced anything new.

In [ ]:
if not pending_raw:
    log.info("Nothing pending. Skipping cleaning.")
    cleaned_batch = []
else:
    cleaned_batch = []
    dropped_empty = 0

    for raw_record in pending_raw:
        cleaned = clean_record(raw_record)
        if cleaned is None or not cleaned.get("job_title"):
            dropped_empty += 1
            continue
        cleaned_batch.append(cleaned)

    log.info("Cleaned %d / %d pending job(s) (%d dropped as empty/unparseable)",
              len(cleaned_batch), len(pending_raw), dropped_empty)

    # Append to the full cleaned history (dedup by job_url = idempotent re-run)
    clean_history = load_json(CLEAN_HISTORY)
    clean_history = dedupe_keep_last(clean_history + cleaned_batch, key="job_url")
    save_json(clean_history, CLEAN_HISTORY)

    # Push this batch onto the queue for notebook 3
    pending_cleaned = load_json(PENDING_CLEANED_QUEUE)
    pending_cleaned = dedupe_keep_last(pending_cleaned + cleaned_batch, key="job_url")
    save_json(pending_cleaned, PENDING_CLEANED_QUEUE)

    # This batch is consumed - clear the raw queue
    save_json([], PENDING_RAW_QUEUE)

    log.info("Clean history total: %d | Pending cleaned queue: %d | Raw queue cleared",
              len(clean_history), len(pending_cleaned))

len(cleaned_batch)